In [1]:
0+9

9

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools import GoogleSerperRun
from langchain_community.utilities import GoogleSerperAPIWrapper
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
load_dotenv(override=True)

search = GoogleSerperRun(api_wrapper=GoogleSerperAPIWrapper())

sandbox=os.path.abspath("sandbox")
os.makedirs(sandbox,exist_ok=True)

model=ChatOpenAI(model="gpt-3.5-turbo")

research = create_deep_agent(
model=model, 
tools=[search],
system_prompt=("you are a reserch analyst. plan your work with todo tool"
"research with search tool and write your findings as a tidy markdown briefing to the file"),
backend = FilesystemBackend(root_dir=sandbox, virtual_mode=True),)


True

In [20]:
brief = """
Our company is planning to move its sales fleet to electric vehicles.
Research the public EV charging landscape in the US: find out roughly how many public charging points there are,
and pick out two major charging networks a fleet could rely on.
Write a one page markdown briefing, with a heading and a short section for each, to the file charging.md.
"""

result=research.invoke({"messages":[{"role":"user","content":brief}]})

print(result["messages"][-1].content)


I have updated the markdown briefing on the public EV charging landscape in the US with information on the approximate number of public EV charging points and two major charging networks. You can view the full briefing in the file [here](sandbox:/charging.md).


In [ ]:
#Sub Agent

research_ev_instructions = """You research one electric vehicle using the search tool and 
return three concise facts that a fleet buyer would care about ,
such as price , range charging"""

overall_instructions = """you write comparison briefing for a company choosing electric vehicles 
for it sales fleet .For each vehicle, delegate the research to your vehicle-research sub-agent, 
then write a markdown comparison to a file,ending with a clear recommendations"""

research_subagent = {
    "name": "vehicle-researcher",
    "description":"Researches a single electric vehicle and returns short list of facts about it",
    "system_prompt": research_ev_instructions,
}

lead = create_deep_agent(
    model=model,
    tools=[search],
    system_prompt=overall_instructions,
    subagents=[research_subagent],
    backend=FilesystemBackend(root_dir=sandbox, virtual_mode=True)
)

mission = """
Compare the Tesla Model Y and the Ford Mustang Mach-E as candidates for our 100-car sales fleet.
Research each vehicle, then write a short markdown comparison with a recommendation to fleet.md.
"""
result = lead.invoke({'messages':[{'role':"user","content":mission}]})
print(result['messages'][-1].content)

The comparison between the Tesla Model Y and the Ford Mustang Mach-E has been completed and the recommendation has been added to fleet.md. You can view the detailed comparison and the recommendation in the fleet.md file.
